In [7]:
# OS interface for interacting with the file system
import os

# Core data manipulation and visualisation libraries
import numpy as np                  # Numerical computing (arrays, math functions)
from scipy.linalg import logm, sqrtm                     # Matrix operations for Riemannian geometry

# ✅ Confirmation
print("Imports done ✅!")

Imports done ✅!


In [8]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
# run the risky operations here

# Generate mean and standard deviation of Riemannian distances for SPD (correlation) matrices of sizes p=5–80
def metropolis_hastings(p, component_index, num_samples=2000, burn_in=200, proposal_std=0.01):
    """
    Generate random correlation matrix components using Metropolis–Hastings sampling on the unit sphere.

    Parameters:
        p (int): Dimension of the full correlation matrix.
        component_index (int): Index of the component being generated (1-based).
        num_samples (int): Number of samples to retain after burn-in.
        burn_in (int): Number of initial samples to discard to allow convergence.
        proposal_std (float): Standard deviation of the Gaussian proposal noise.

    Returns:
        numpy.ndarray: Array of shape (num_samples, dim) containing generated unit vectors.
    """
    dim = p - component_index + 1  # Effective dimension of the current component
    samples = np.zeros((burn_in + num_samples + 1, dim))

    # Initialise with a random unit vector; ensure first component is non-negative
    current = np.random.randn(1, dim)
    current[0, 0] = np.abs(current[0, 0])
    current = current / np.linalg.norm(current)

    # Run Metropolis–Hastings chain
    for t in range(burn_in + num_samples + 1):
        # Propose a new vector with small Gaussian perturbation
        proposal = current + np.random.normal(scale=proposal_std, size=(1, dim))
        proposal = proposal / np.linalg.norm(proposal)  # Renormalise to unit length

        # Compute acceptance ratio (ensuring positivity of first component)
        delta = np.random.uniform(0, 1)
        acceptance_ratio = (proposal[0, 0] / current[0, 0])**component_index if proposal[0, 0] >= 0 else 0

        # Accept or reject proposal
        if delta < acceptance_ratio:
            current = proposal

        samples[t, :] = current.flatten()

    # Return only the post-burn-in samples
    return samples[burn_in + 1:, :]


def random_correlation_matrix(p):
    """
    Construct a random correlation matrix of size p × p using Metropolis–Hastings sampling.

    Each row of the upper-triangular matrix U is sampled sequentially, and the
    final correlation matrix is obtained as C = U Uᵀ, normalised to have unit diagonal.

    Parameters:
        p (int): Dimension of the correlation matrix.

    Returns:
        numpy.ndarray: Random correlation matrix of size p × p.
    """
    U = np.zeros((p, p))
    for i in range(p):
        # For each row i, sample the corresponding unit vector and use the last draw
        U[i, i:] = metropolis_hastings(p, i + 1)[-1, :]

    # Compute symmetric product and normalise diagonal entries to 1
    C = U @ U.T
    D = np.sqrt(np.diag(C))
    correlation_matrix = C / np.outer(D, D)
    return correlation_matrix


def distance_riemann(A, B):
    """
    Compute the affine-invariant Riemannian distance between two SPD matrices A and B.

    Formula:
        d(A, B) = || log( A^{-1/2} B A^{-1/2} ) ||_F

    Parameters:
        A (numpy.ndarray): First SPD matrix.
        B (numpy.ndarray): Second SPD matrix.

    Returns:
        float: The affine-invariant Riemannian distance between the two matrices.
    """
    sqrt_A = sqrtm(A)
    inv_sqrt_A = np.linalg.inv(sqrt_A)
    C = inv_sqrt_A @ B @ inv_sqrt_A
    log_C = logm(C)
    return np.linalg.norm(log_C, 'fro')


def expected_distance(matrix_size, num_pairs=100):
    """
    Estimate the expected Riemannian distance (mean ± std) between pairs of
    random correlation matrices of a given size.

    Parameters:
        matrix_size (int): Dimension of the correlation matrices.
        num_pairs (int): Number of pairs of matrices to generate.

    Returns:
        tuple:
            - float: Mean of the estimated distances.
            - float: Standard deviation of the estimated distances.
    """
    distances = []
    for _ in range(num_pairs):
        A = random_correlation_matrix(matrix_size)
        B = random_correlation_matrix(matrix_size)
        d = distance_riemann(A, B)
        distances.append(d)

    return np.mean(distances), np.std(distances)



In [9]:
# Main loop: compute mean and standard deviation of average riemannian distances between 
# random matrices ranging from 5 to 81 dimensions
mean_d = []
std_d = []
for n in range(5,81):
    mean_tmp, std_tmp = expected_distance(n, num_pairs=100)
    print(f"{n}x{n}: Mean: {mean_tmp:.4f}, Standard Deviation: {std_tmp:.4f}")

    mean_d.append(mean_tmp)
    std_d.append(std_tmp)
    

5x5: Mean: 5.3036, Standard Deviation: 1.7309
6x6: Mean: 6.1436, Standard Deviation: 1.5292


In [ ]:
#values reported in the jupyther notebook
stim_dist = [0.8543, 0.8024, 0.8918, 0.7506, 0.8326, 0.8286, 0.7508, 0.7120, 0.8152, 0.7978,
             0.5823, 0.7944, 0.6782, 0.4742, 0.3286, 0.4824, 0.7652, 0.7087, 0.6085, 0.6523]


In [ ]:

for i in range(20):
    #load covariance matrices
    filename = f"data/calcium_imaging/cov_matrices/{i+1}_10Hz_cov_norm_pre.csv"
    filename2 = f"data/calcium_imaging/cov_matrices/{i+1}_10Hz_cov_norm_stim.csv"
    A = np.loadtxt(filename, delimiter='\t', dtype=float)
    B = np.loadtxt(filename2, delimiter='\t', dtype=float)
    
    #normalize A to correlation matrix
    v = np.sqrt(np.diag(A))
    outer_v = np.outer(v, v)
    A2 = A / outer_v
    A2[A == 0] = 0

    #normalize B to correlation matrix
    v = np.sqrt(np.diag(B))
    outer_v = np.outer(v, v)
    B2 = B / outer_v
    B2[B == 0] = 0

    #compute normalised riemannian distance
    dist = distance_riemann(A2,B2)/mean_d[A.shape[0]]
